# Problem Statement 2: Data Wrangling II
**Objective:** Create an Academic Performance dataset and handle missing values, outliers, and apply data transformations.

## Step 1: Import Libraries and Create Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)

# Create Academic Performance dataset
n = 100
data = {
    'StudentID': range(1, n + 1),
    'Name': [f'Student_{i}' for i in range(1, n + 1)],
    'Age': np.random.randint(17, 25, n).astype(float),
    'Gender': np.random.choice(['Male', 'Female', None], n, p=[0.45, 0.45, 0.10]),
    'MathScore': np.random.randint(40, 100, n).astype(float),
    'EnglishScore': np.random.randint(35, 100, n).astype(float),
    'ScienceScore': np.random.randint(30, 100, n).astype(float),
    'Attendance': np.random.uniform(50, 100, n),
    'StudyHoursPerDay': np.random.uniform(1, 10, n)
}

df = pd.DataFrame(data)

# Introduce missing values manually
df.loc[np.random.choice(n, 10, replace=False), 'Age'] = np.nan
df.loc[np.random.choice(n, 8, replace=False), 'MathScore'] = np.nan
df.loc[np.random.choice(n, 5, replace=False), 'Attendance'] = np.nan

# Introduce outliers
df.loc[0, 'MathScore'] = 200   # impossible score
df.loc[1, 'StudyHoursPerDay'] = 25  # impossible hours

print("Dataset created!")
print("Shape:", df.shape)
df.head(10)

## Step 2: Scan for Missing Values and Inconsistencies

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nMissing % per column:")
print((df.isnull().sum() / len(df)) * 100)

In [ ]:
# Handle missing values
# Age: fill with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# MathScore: fill with mean
df['MathScore'].fillna(df['MathScore'].mean(), inplace=True)

# Attendance: fill with mean
df['Attendance'].fillna(df['Attendance'].mean(), inplace=True)

# Gender: fill with mode
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)

print("Missing values after handling:")
print(df.isnull().sum())

## Step 3: Scan for Outliers Using IQR Method
The IQR (Interquartile Range) method detects outliers that fall below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.

In [ ]:
numeric_cols = ['Age', 'MathScore', 'EnglishScore', 'ScienceScore', 'Attendance', 'StudyHoursPerDay']

print("Outlier Detection using IQR Method:\n")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outlier(s) | Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")

In [ ]:
# Visualize outliers with boxplots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
    axes[i].set_ylabel('Values')

plt.suptitle('Boxplots for Outlier Detection', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Handle outliers: Cap/clip values to IQR bounds (Winsorization)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print("Outliers handled using Winsorization (clipping to IQR bounds).")
print(df[numeric_cols].describe())

## Step 4: Data Transformation
**Purpose:** The `StudyHoursPerDay` variable might be right-skewed. We apply a **log transformation** to reduce skewness and make it closer to a normal distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before transformation
axes[0].hist(df['StudyHoursPerDay'], bins=15, color='skyblue', edgecolor='black')
axes[0].set_title('StudyHoursPerDay - Before Log Transform')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Frequency')

# Apply log transformation
df['StudyHours_Log'] = np.log1p(df['StudyHoursPerDay'])  # log1p = log(1+x) to handle 0

# After transformation
axes[1].hist(df['StudyHours_Log'], bins=15, color='salmon', edgecolor='black')
axes[1].set_title('StudyHoursPerDay - After Log Transform')
axes[1].set_xlabel('Log(Hours)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"Skewness before: {df['StudyHoursPerDay'].skew():.4f}")
print(f"Skewness after:  {df['StudyHours_Log'].skew():.4f}")
print("\nLog transformation reduced skewness, making distribution more normal.")

In [ ]:
# Compute TotalScore and Grade
df['TotalScore'] = df['MathScore'] + df['EnglishScore'] + df['ScienceScore']
df['Average'] = df['TotalScore'] / 3

def assign_grade(avg):
    if avg >= 80: return 'A'
    elif avg >= 65: return 'B'
    elif avg >= 50: return 'C'
    else: return 'D'

df['Grade'] = df['Average'].apply(assign_grade)

print("Final Dataset:")
df[['StudentID', 'MathScore', 'EnglishScore', 'ScienceScore', 'TotalScore', 'Average', 'Grade']].head(10)